# Step 2 — Detecting blinks from a single number (EAR)

Step 1 confirmed we can reliably find eye landmarks. Now: can we turn
those into a blink detector?

The classic trick is the **eye aspect ratio (EAR)**: roughly, how "open"
the eye looks, as a single number per frame. It's cheap to compute and
drops sharply when the eye closes — tempting to use with a simple cutoff
("below X = closed"). Spoiler, from the project notes: *that cutoff isn't
as simple as it looks.* This notebook builds the simple version first —
`step_05` tests it properly, against real hand labels and a small model.


## At a glance

| | |
|---|---|
| **Workshop step** | 2 of 6 |
| **Needs in this folder** | your blink clip(s), named `blinks_<name>.mp4` (see `README.md`). In a group, one per person. |
| **Produces** | `ear_blinks_<name>.csv` for each blink clip (one EAR value per frame). The labelling tool in step 4 reads these. |
| **Run time** | roughly 10 to 20 seconds per clip on a normal laptop |
| **Next** | `step_03_gaze_head_features.ipynb` |

**How to run:** in the menu choose *Run > Restart Kernel and Run All Cells*, or press Shift+Enter on each cell from the top down. Do not skip cells: later cells use variables made by earlier ones.

MediaPipe prints warnings in red or pink (libEGL, TensorFlow Lite, "feedback manager", protobuf). They are harmless, ignore them.

## The eye aspect ratio

For six eye landmarks (outer corner $p_1$, two upper-lid points $p_2, p_3$,
inner corner $p_4$, two lower-lid points $p_5, p_6$):

$$EAR = \frac{|p_2 - p_6| + |p_3 - p_5|}{2 \cdot |p_1 - p_4|}$$

- The numerator is roughly the eye's **height** (two vertical lid-to-lid
  distances, averaged).
- The denominator is the eye's **width** (corner to corner), doubled to
  match the two height terms.
- An open eye sits around 0.2–0.35 depending on gaze direction (more on
  that below). A fully closed eye pushes this toward 0, since the lids
  are touching.

Dividing by width instead of using raw pixel height matters because it
makes EAR roughly independent of how close someone is to the camera.


In [ ]:
%matplotlib inline
import glob
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mediapipe as mp

# Every blink clip in this folder (naming rule: see README.md).
# In a group there is one per person, and each gets its own threshold.
CLIPS = sorted(glob.glob("blinks_*.mp4"))
if not CLIPS:
    raise SystemExit("No blink clips found. Save your recording here as blinks_<name>.mp4 (see README.md).")
print("Blink clips:", CLIPS)

# six landmarks per eye, ordered: outer corner, 2 upper lid, inner corner, 2 lower lid
R_EYE = [33, 160, 158, 133, 153, 144]
L_EYE = [362, 385, 387, 263, 373, 380]


def eye_aspect_ratio(p):
    """Eye height divided by eye width. Drops toward 0 when the eye closes."""
    vertical = np.linalg.norm(p[1] - p[5]) + np.linalg.norm(p[2] - p[4])
    horizontal = np.linalg.norm(p[0] - p[3])
    return vertical / (2.0 * horizontal)

**Before running this on a whole clip:** sketch (on paper, or just in
your head) what you expect the EAR signal to look like over your whole clip
(about 30 seconds) if you blink 15 to 20 times. Flat with occasional sharp dips? Noisy
throughout? Slowly drifting?


In [ ]:
def ear_for_clip(path):
    """Run face mesh over one clip. Returns (EAR per frame, frames per second).

    Frames where no face was found get NaN ("not a number"), so a missed
    frame is never mistaken for a real measurement.
    """
    mesh = mp.solutions.face_mesh.FaceMesh(
        max_num_faces=1, refine_landmarks=True,
        min_detection_confidence=0.5, min_tracking_confidence=0.5,
    )
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise SystemExit(f"Could not open {path}")
    fps = cap.get(cv2.CAP_PROP_FPS)

    ears = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        h, w = frame.shape[:2]
        result = mesh.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        if result.multi_face_landmarks:
            lm = result.multi_face_landmarks[0].landmark
            pts = np.array([[p.x * w, p.y * h] for p in lm])  # pixel coordinates
            # TRY IT: this averages the two eyes. Keep only the right or only
            # the left eye instead. Do the two eyes always blink the same way?
            ears.append((eye_aspect_ratio(pts[R_EYE]) + eye_aspect_ratio(pts[L_EYE])) / 2)
        else:
            ears.append(np.nan)
    cap.release()
    mesh.close()
    return np.array(ears), fps


ear_by_clip = {}   # clip name without ".mp4" -> (EAR per frame, time of each frame, fps)
for clip in CLIPS:
    ears, fps = ear_for_clip(clip)
    times = np.arange(len(ears)) / fps
    ear_by_clip[os.path.splitext(clip)[0]] = (ears, times, fps)
    print(f"{clip}: {len(ears)} frames at {fps:.1f} fps ({len(ears)/fps:.1f} s)")
    print(f"   EAR range: {np.nanmin(ears):.3f} to {np.nanmax(ears):.3f}, median {np.nanmedian(ears):.3f}")

### Plot the raw EAR signal

In [ ]:
fig, axes = plt.subplots(len(ear_by_clip), 1, figsize=(12, 3 * len(ear_by_clip)), squeeze=False)
for ax, (name, (ears, times, fps)) in zip(axes[:, 0], ear_by_clip.items()):
    ax.plot(times, ears, lw=1)
    ax.set_ylabel("EAR")
    ax.set_title(f"{name}: raw EAR signal -- where would you guess the blinks are?")
axes[-1, 0].set_xlabel("time (s)")
plt.tight_layout()
plt.show()

## Turning this into a yes/no blink detector

The simplest approach: pick a cutoff, call anything below it "closed."
But whose cutoff? A fixed number like 0.2 assumes everyone's open-eye EAR
looks the same — it doesn't (eye shape, camera angle, and *gaze direction*
all shift it; `step_05` comes back to this).

Instead, we calibrate the threshold **to each clip itself**:
1. Take the 90th percentile of EAR as "normal open" (robust to a few
   already-closed frames pulling the max down).
2. Multiply by 0.75 as a margin — closed should be well below normal
   open, not just slightly below it.

This is still a *global* threshold (one number for a whole clip) — it
doesn't yet account for EAR changing *within* a clip. That gap is exactly
what `step_05`'s model will be asked to do better on.


In [ ]:
# ---------------------------------------------------------------------
# TRY IT: these four numbers ARE the blink detector. Change one at a
# time, rerun this cell and the plot below, and watch the orange bands.
#   BASELINE_PERCENTILE  what counts as "normal open". Try 50 (the median) or 99.
#   THRESHOLD_FACTOR     how far below normal counts as closed. Try 0.6 and 0.9.
#   MIN_BLINK_FRAMES     shortest run of closed frames kept as a blink. Try 1.
#   MAX_BLINK_SECONDS    longest run still called a blink. Try 0.3 and 2.0.
# Which setting finds the blinks you can see in the plot, and which adds
# false ones? Is the best setting the same for everyone in your group?
# ---------------------------------------------------------------------
BASELINE_PERCENTILE = 90
THRESHOLD_FACTOR = 0.75
MIN_BLINK_FRAMES = 2
MAX_BLINK_SECONDS = 0.8


def detect_blinks(ears, fps):
    """Fixed-threshold blink detector for one clip.

    Returns (baseline, threshold, blinks), where blinks is a list of
    (start frame, end frame) pairs, end not included.
    """
    baseline = np.nanpercentile(ears, BASELINE_PERCENTILE)
    threshold = THRESHOLD_FACTOR * baseline
    closed = np.nan_to_num(ears, nan=1.0) < threshold   # a missed frame counts as "open"

    # group consecutive closed frames into blink events; keep plausible durations
    blinks, start = [], None
    for i, c in enumerate(closed):
        if c and start is None:
            start = i
        if start is not None and (not c or i == len(closed) - 1):
            end = i + 1 if c else i
            n_frames = end - start
            # drop single-frame noise, and anything too long to be a blink
            if n_frames >= MIN_BLINK_FRAMES and n_frames / fps <= MAX_BLINK_SECONDS:
                blinks.append((start, end))
            start = None
    return baseline, threshold, blinks


detected_by_clip = {}   # clip name -> (threshold, blinks)
for name, (ears, times, fps) in ear_by_clip.items():
    baseline, threshold, blinks = detect_blinks(ears, fps)
    detected_by_clip[name] = (threshold, blinks)
    print(f"{name}: baseline EAR {baseline:.3f} | threshold {threshold:.3f} | detected blinks: {len(blinks)}")
    for k, (s, e) in enumerate(blinks, 1):
        print(f"  {k:2d}. start {times[s]:6.2f}s | duration {(e - s) / fps:.2f}s")

### Plot the result

Blue line: EAR. Red dashed line: the threshold. Orange bands: stretches the rule calls a blink.

In [ ]:
fig, axes = plt.subplots(len(ear_by_clip), 1, figsize=(12, 4 * len(ear_by_clip)), squeeze=False)
for ax, (name, (ears, times, fps)) in zip(axes[:, 0], ear_by_clip.items()):
    threshold, blinks = detected_by_clip[name]
    ax.plot(times, ears, lw=1, label="EAR")
    ax.axhline(threshold, color="r", ls="--", label="blink threshold")
    for s, e in blinks:
        ax.axvspan(times[s], times[e - 1], color="orange", alpha=0.3)
    ax.set_ylabel("eye aspect ratio")
    ax.set_title(name)
    ax.legend(loc="upper right")
axes[-1, 0].set_xlabel("time (s)")
plt.tight_layout()
plt.show()

### Save the per-frame EAR

Do not skip this cell. `step_04_label_intervals.py` reads `ear_blinks_<name>.csv` to show the EAR value next to each video frame.

In [ ]:
for name, (ears, times, fps) in ear_by_clip.items():
    out = f"ear_{name}.csv"
    pd.DataFrame({"frame": np.arange(len(ears)), "time_s": times.round(4), "ear": ears}).to_csv(out, index=False)
    print(f"Saved {out} ({len(ears)} rows)")

## Discuss — would this threshold survive a harder test?

This rule may look convincing on your clip. But:
- **Does the orange shading actually line up with real blinks**, or does
  it also catch moments where the person just looked down (lowering EAR
  without closing their eyes)?
- The threshold is *relative to this clip's own 90th percentile*. What
  happens if you ran this exact code on a clip where someone spends more
  time looking down — would the baseline (and therefore the threshold)
  shift, and in which direction?
- With more than one clip: are the baselines (and so the thresholds)
  the same for everyone in your group? If not, what does that say about
  using one fixed number such as 0.2 for all people?
- `step_04` asks you to actually hand-label real blink intervals by
  watching the footage. `step_05` then checks this exact rule against
  those hand labels, and compares it to a model that also sees gaze
  direction. Keep today's blink count and threshold value in mind — you'll
  revisit this exact number later.

**Next:** `step_03` builds the gaze and head features that `step_05`/
`step_06` use to try to do better than a plain EAR cutoff.
